# Data Cleaning Workflow

This notebook starts the reproducible data-cleaning workflow for the selected targets and the predictor datasets listed in the literature CSV. Reusable download logic lives in `2_data/scripts/raw_data_download.py`; this notebook imports the script, runs the workflow, and displays the resulting manifest.

## Optional Colab Setup

This cell is safe for local use. It only performs setup when the notebook runs inside Google Colab.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Colab detected. Mount your repository and set ROOT_DIR in the next cell if needed.")
else:
    print("Local environment detected; skipping Colab setup.")

## Repository Setup

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "0_organization").exists() and (path / "2_data").exists():
            return path
    raise RuntimeError("Could not find repository root. Run this notebook from inside the repository.")


ROOT_DIR = find_repo_root(Path.cwd())
SCRIPT_DIR = ROOT_DIR / "2_data" / "scripts"

if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

ROOT_DIR

## Literature CSV-Based Raw Download Plan

The predictor download plan uses `1_literature_review/Managerial AI- literature review - List 1.csv` as the primary source. The manifest is generated by `run_raw_download`; do not edit it by hand. If a future literature source lacks a machine-readable endpoint, extend `2_data/scripts/raw_data_download.py` so the missing source remains reproducible.

In [ ]:
import pandas as pd
from data_common import RAW_PREDICTORS_V1_DIR
from raw_data_download import (
    LITERATURE_CSV_PATH,
    build_literature_predictor_download_plan,
    build_raw_download_plan,
    run_raw_download,
)

START_YEAR = 1990
END_YEAR = 2024
RAW_OUTPUT_DIR = RAW_PREDICTORS_V1_DIR

literature_plan = pd.DataFrame(
    build_literature_predictor_download_plan(LITERATURE_CSV_PATH, START_YEAR, END_YEAR)
)
download_plan = pd.DataFrame(build_raw_download_plan(START_YEAR, END_YEAR, LITERATURE_CSV_PATH))

literature_plan[[
    "dataset_id",
    "variable",
    "source_variable",
    "status",
    "download_method",
    "literature_rows",
    "literature_predictors",
]]

## Download Raw Files

Running this cell writes source files to `2_data/raw/predictorsv1/` and writes `raw_download_manifest.csv` in the same directory with source URLs, source status, download dates, file paths, and source notes. Both the raw files and manifest are code-generated artifacts.

In [ ]:
manifest = run_raw_download(start_year=START_YEAR, end_year=END_YEAR, raw_dir=RAW_OUTPUT_DIR)
manifest[[
    "dataset_id",
    "variable",
    "role",
    "status",
    "rows",
    "columns",
    "file_format",
    "file_path",
    "notes",
]]

## Download Status Checks

The status table is a reproducibility check. A successful raw download run should have only `downloaded` rows; any future non-downloaded row means the downloader should be extended before downstream cleaning proceeds.

In [ ]:
status_counts = (
    manifest.groupby(["status", "dataset_id"], dropna=False)
    .size()
    .reset_index(name="entries")
    .sort_values(["status", "dataset_id"])
)

unresolved_sources = manifest[manifest["status"].ne("downloaded")][[
    "variable",
    "source",
    "source_url",
    "status",
    "notes",
]]

display(status_counts)
display(unresolved_sources)

## Raw Source Audit

This section validates each downloaded raw source and writes one consolidated audit table to `2_data/processed/raw_predictor_audit.csv`. The table combines source reliability checks, file shape checks, checksums, and country/entity-year coverage. Detailed panel and by-year objects are kept in memory for figures rather than saved as intermediate CSV files. A `confirmed` status means the file is present and machine-readable; it does not mean the variable is already clean enough for modeling.


In [ ]:
from raw_coverage_diagnostics import run_coverage_diagnostics

coverage_outputs = run_coverage_diagnostics(raw_dir=RAW_OUTPUT_DIR)
predictor_audit = coverage_outputs["predictor_audit"]
predictor_audit_path = coverage_outputs["predictor_audit_path"]
reliability_audit = coverage_outputs["reliability_audit"]
coverage_summary = coverage_outputs["coverage_summary"]
coverage_by_year = coverage_outputs["coverage_by_year"]
country_coverage_summary = coverage_outputs["country_coverage_summary"]
figure_paths = coverage_outputs["figure_paths"]

predictor_audit[[
    "dataset_id",
    "variable",
    "source_domain",
    "file_format",
    "actual_rows",
    "actual_columns",
    "reliability_status",
    "entities_with_data",
    "years_with_data",
    "first_year",
    "last_year",
    "source_series_count",
]]


## Raw Coverage Diagnostics

Coverage is measured at the raw country/entity-year level after excluding standard aggregate area codes where country codes are available. RISE is split into the two selected World Bank Data360 groups: Renewable Energy (`WB_RISE_RE_*`) and Energy Efficiency (`WB_RISE_EE_*`); the summary reports each group's retained sub-indicator count in `source_series_count`. EPU is converted from the monthly workbook to annual country-series means and clipped to the requested 1990-2024 window.


In [ ]:
coverage_summary[[
    "variable_group",
    "dataset_id",
    "variable",
    "entities_with_data",
    "years_with_data",
    "first_year",
    "last_year",
    "non_missing_observations",
    "source_series_count",
    "country_code_complete_share",
]].sort_values(["variable_group", "entities_with_data", "years_with_data"], ascending=[True, False, False])


## Coverage Figures

The variable ranking combines covered country/entity counts with a first-to-last covered year span in one panel. The country ranking orders 3-letter source-code countries/entities by their share of non-missing variable-year opportunities. These figures are generated by this notebook through `raw_coverage_diagnostics.py` and written to `4_analysis/figures/predictorsv1/`.


In [ ]:
from IPython.display import Image, display

for key in ["variable_country_rank_png", "country_coverage_rank_png"]:
    display(Image(filename=figure_paths[key]))


## Lagged Analysis-Base Panel Cleaning

This section creates lagged country-year analysis-base panels from the downloaded raw files. The reusable logic lives in `2_data/scripts/model_panel_cleaning.py`. Each predictor is converted into two lagged features: `x_lag1` and `x_lag1_3_mean`. The no-imputation panels are the primary prediction-safe inputs. The linear-interpolated panels are kept only as retrospective sensitivity outputs because full-series interpolation can use future endpoints before lag construction.


In [ ]:
from model_panel_cleaning import run_model_panel_cleaning

model_panel_outputs = run_model_panel_cleaning(raw_dir=RAW_OUTPUT_DIR)
model_panel_coverage = model_panel_outputs["coverage_summary"]
model_panel_imputation = model_panel_outputs["imputation_summary"]
model_panel_variable_map = model_panel_outputs["variable_map"]

coverage_columns = [
    "panel_id",
    "imputation",
    "recommended_use",
    "prediction_safe",
    "anchor_variable",
    "countries",
    "rows",
    "target_non_missing",
    "lag1_complete_rows",
    "target_lag1_complete_rows",
    "target_lag1_complete_countries",
    "lag1_3_mean_complete_rows",
    "target_lag1_3_mean_complete_rows",
    "target_lag1_3_mean_complete_countries",
    "lag1_3_mean_complete_target_missing_rows",
    "imputed_values_total",
]
model_panel_coverage[coverage_columns]


The rows above distinguish feature completeness from actual supervised-learning availability. For example, `lag1_3_mean_complete_rows` counts rows with complete three-year lagged features, while `target_lag1_3_mean_complete_rows` additionally requires a non-missing target in the same country-year.


## Panel Variable Map

The variable map links each target or predictor back to its raw file and source code. It also records the anchor variable for each panel and the RISE score selection rule.

In [ ]:
model_panel_variable_map[[
    "panel_id",
    "variable",
    "role",
    "is_anchor",
    "dataset_id",
    "source_variable",
    "raw_file",
    "rise_selection_rule",
    "predictor_window_start",
    "predictor_window_end",
]]


## Generated Panel Files

The cleaning script writes eight candidate model panels plus two metadata tables to `2_data/processed/`. `model_panel.csv` is intentionally not created yet; the final single modeling input should be selected after comparing coverage and missingness.

In [ ]:
panel_file_summaries = []
for file_name in model_panel_outputs["panel_paths"]:
    panel_path = ROOT_DIR / "2_data" / "processed" / file_name
    panel_data = pd.read_csv(panel_path)
    panel_file_summaries.append(
        {
            "file_name": file_name,
            "rows": len(panel_data),
            "columns": len(panel_data.columns),
            "countries": panel_data["country_code"].nunique(),
            "first_year": int(panel_data["year"].min()),
            "last_year": int(panel_data["year"].max()),
        }
    )

pd.DataFrame(panel_file_summaries)

## Cleaning Output Note

The main panel uses `fossil_energy_share` as the anchor variable and raw predictor years 1996-2022, which map to target years 1999-2023 for the three-year lagged mean. Submodel anchors and raw windows are selected automatically from the common predictor coverage in each submodel. All outputs use World Bank/OECD-style 3-letter country codes for merges; source-specific codes such as `XKX` are retained when present.